In [1]:
from microsim.allen import Specimen
from microsim.util import ndview
from tifffile import imwrite
import numpy as np

spec = Specimen.fetch(555241040)
swc = spec.neuron_reconstructions[0].binary_mask(voxel_size=1, scale_factor=3)

In [ ]:
from microsim.allen import Specimen
from microsim.util import ndview
from tifffile import imwrite
import numpy as np

for id in range(10):
    spec = Specimen.fetch(555241040)
    swc = spec.getSomaLocation()
    grid = neuron_reconstructions[0].binary_mask(voxel_size=1, scale_factor=3, empty_grid=grid)
# get_all_coordinates(), rotate(), translate(), scale(), get_all_coordinates()

In [1]:
from microsim.allen import Specimen
from microsim.util import ndview
from tifffile import imwrite
import numpy as np

spec = Specimen.fetch(555241040)

swc = spec.soma_location()
print(swc)

spec.new_extent(dim=512)
swc = spec.soma_location()
print(swc)

# spec.rotated_soma((20.0,35.0,545.0))
# swc = spec.soma_location()
# print(swc)

# spec.scaled_soma(2.0)
# swc = spec.soma_location()
# print(swc)






[Compartment(id=1, t=1, x=416.4583, y=333.0401, z=25.8964, r=6.5677, c=-1)]
[Compartment(id=1, t=1, x=272.05283803564816, y=255.38367388072447, z=80.70168998287, r=6.5677, c=-1)]


In [ ]:
from microsim.allen import Specimen
from microsim.util import ndview
from tifffile import imwrite
import numpy as np

spec = Specimen.fetch(555241040)
coords = spec.coords_location()
print(len(coords[1]))
for i in range(1):
    print(coords[1][i])
spec.offset_coords((3.0, 4.0, 5.0))
for i in range(1):
    print(coords[1][i])    

In [1]:
from microsim.allen import Specimen
from microsim.util import ndview
from tifffile import imwrite
import numpy as np

spec = Specimen.fetch(555241040)
spec.augmented_masks(offset=(3.0, 4.0, 5.0), rotation=(20.0,35.0,545.0), scale=2.0)



In [ ]:
import numpy as np
from scipy.ndimage import rotate
import random
import numpy as np
from scipy.ndimage import gaussian_filter

#add gaussian blur to the line
for i in range(swc.shape[0]):
    swc[i] = gaussian_filter(swc[i], sigma=1.0)

def rotate_randomly(array):
    # Generate random angle between 0 and 360 degrees
    angle = random.uniform(50, 300)
    
    # Choose a random axis for rotation
    axes = [(0, 1), (0, 2), (1, 2)]
    axis = random.choice(axes)
    print(axis, angle)
    
    # Rotate the array
    rotated_array = rotate(array, angle, axes=axis, reshape=False, mode='reflect')
    
    return rotated_array

#rotate the swc 5 times and add the results to new array
rotated_swcs = np.zeros_like(swc)
for i in range(3):
    rotated_swcs += rotate_randomly(swc)

#save the rotated swcs as a tiff
imwrite("swc_rotated.tif", rotated_swcs)


In [ ]:
import itk
from itkwidgets import view
image = itk.image_view_from_array(rotated_swcs)
view(image)

In [ ]:
from microsim.allen import Specimen
from microsim.util import ndview
from tifffile import imwrite
import numpy as np
import numpy as np
from scipy.ndimage import rotate
import random
import numpy as np
from scipy.ndimage import gaussian_filter
from tqdm import tqdm


def rotate_randomly(array):
    # Generate random angle between 0 and 360 degrees
    angle = random.uniform(50, 300)

    # Choose a random axis for rotation
    axes = [(0, 1), (0, 2), (1, 2)]
    axis = random.choice(axes)
    print(axis, angle)

    # Rotate the array
    rotated_array = rotate(array,
                           angle,
                           axes=axis,
                           reshape=False,
                           mode='reflect')

    return rotated_array

#read a csv file with specimen ids
specimen_id = np.loadtxt("specimen_ids.csv", delimiter=",", dtype=int)

min_x = 20000
min_y = 20000
for i in tqdm(range(len(specimen_id))):
    spec = Specimen.fetch(specimen_id[i])
    swc = spec.neuron_reconstructions[0].binary_mask(voxel_size=1, scale_factor=3)
    if swc.shape[1] < min_x:
        min_x = swc.shape[1]
    if swc.shape[2] < min_y:
        min_y = swc.shape[2]
print(min_x, min_y)

In [ ]:
#read tiff files from the folder and print the shape of the arrays
import os
from tifffile import imread
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
from scipy.ndimage import rotate

path = f"/group/jug/Anirban/Datasets/AllenNeuron/all"
files = os.listdir(path)
#create a dictionary to store the shapes of the arrays with specimen ids as keys
files.sort()
minimum_shape = 512

def pad_crop(swc, minimum_shape):
    pad_x = ((minimum_shape - swc.shape[1]) // 2) + 1 if swc.shape[1] < minimum_shape else 0
    pad_y = ((minimum_shape - swc.shape[2]) // 2) + 1 if swc.shape[2] < minimum_shape else 0
    swc = np.pad(swc, ((0, 0), (pad_x, pad_x), (pad_y, pad_y)), mode='constant', constant_values=0)
    return swc

def pad_remove_channels(swc, channels):
    if swc.shape[0] < channels:
        swc = np.pad(swc, ((0, channels - swc.shape[0]), (0, 0), (0, 0)), mode='constant', constant_values=0)
    elif swc.shape[0] > channels:
        #center crop
        swc = swc[(swc.shape[0] - channels) // 2:(swc.shape[0] + channels) // 2, :, :]
    elif swc.shape[0] == channels:
        return swc
    return swc

def find_coordinates_crop(swc, minimum_shape = 512, minimum_count = 0.8):
    swc_max = np.max(swc, axis=0, keepdims=True)
    for _ in range(50):
        random_x = random.randint(0, swc_max.shape[1] - minimum_shape)  
        random_y = random.randint(0, swc_max.shape[2] - minimum_shape)
        crop = swc_max[:, random_x:random_x + minimum_shape, random_y:random_y + minimum_shape]
        count_nonzero = np.count_nonzero(crop)
        if count_nonzero > minimum_count * minimum_shape * minimum_shape:
            break
    swc = swc[:, random_x:random_x + minimum_shape, random_y:random_y + minimum_shape]
    return swc

def rotate_randomly(array):
    angle = random.uniform(50, 150)
    axes = [(0, 1), (0, 2), (1, 2)]
    axis = random.choice(axes)
    rotated_array = rotate(array,angle, axes=axis, reshape=False, mode='reflect')

    return rotated_array

for i in tqdm(range(1)):
    swc = imread(f"{path}/{files[i]}")
    swc = pad_crop(swc, minimum_shape)
    swc = find_coordinates_crop(swc)
    for batch in tqdm(range(9), leave=False):
        rotated_swcs = np.zeros_like(swc)
        rotated_swcs += rotate_randomly(swc)
        for _ in range(2):
            random_specimen = random.choice(files[:i] + files[i + 1:])
            new_swc = imread(f"{path}/{random_specimen}")
            new_swc = pad_remove_channels(new_swc, swc.shape[0])
            new_swc = pad_crop(new_swc, minimum_shape)
            new_swc = find_coordinates_crop(new_swc)
            rotated_swcs += rotate_randomly(new_swc)
            
    
    


In [ ]:
from microsim.allen import Specimen
from microsim.util import ndview
from tifffile import imwrite
import numpy as np
import numpy as np
from scipy.ndimage import rotate
import random
import numpy as np
from scipy.ndimage import gaussian_filter
from tqdm import tqdm


specimen_ids = np.loadtxt("specimen_ids.csv", delimiter=",", dtype=int)
spec = Specimen.fetch(specimen_ids[0])
swc = spec.neuron_reconstructions[0].binary_mask(voxel_size=2, scale_factor=3)

In [ ]:
import itk
from itkwidgets import view
image = itk.image_view_from_array(swc)
view(swc)